# 🛰️ Minor Project: Benchmarking the Two Latest 2026 Models

### **Project Focus:**
Head-to-head comparison between the two latest cutting-edge **2026 architectures**:
- **MMFormer** (2026, IEEE TGRS) — Macro-Micro Transformer
- **R2Mamba** (2026, IEEE JSTARS) — Route-Reliability Mamba / State Space Model

**Datasets:** `Indian_Pines` + `Pavia University`  
**Evaluation:** 3 seeded runs $\rightarrow$ **Mean ± Std** (OA, AA, Kappa) + Arranged Classification Maps

### ⚡ Cell 1: Check GPU Environment

In [ ]:
import torch
print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device      :", torch.cuda.get_device_name(0))
    print("VRAM Available  :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("⚠️ Please enable GPU in settings (Accelerator -> GPU T4 x2)")

### 📥 Cell 2: Clone Repository & Navigate

In [ ]:
!git clone https://github.com/Tanishq251/Hyperspectral-Image-Models.git
import os
target_dir = "/kaggle/working/Hyperspectral-Image-Models" if os.path.exists("/kaggle/working") else "Hyperspectral-Image-Models"
os.chdir(target_dir)
print("Working directory:", os.getcwd())

### ⚙️ Cell 3: Install Core Framework Dependencies

In [ ]:
!pip install -r requirements.txt -q
!pip install -r requirements-optional.txt -q
print("✅ Core packages installed!")

### 🐍 Cell 4: Install Mamba / State Space Model Kernels (Optional but Recommended)

> This builds the CUDA accelerated kernels for `MambaHSI` and other SSM models. Takes ~3–6 minutes.

In [ ]:
print("Installing CUDA kernels for Mamba (causal-conv1d & mamba-ssm)...")
!pip install "causal-conv1d>=1.4.0" --no-build-isolation -q
!pip install "mamba-ssm>=2.2.2" --no-build-isolation -q
print("✅ Mamba kernels installed successfully!")

### 📄 Cell 5: Create Main Experiment Configuration (`config/project_main.yaml`)

In [ ]:
config = """
# ============================================================================
#  MINOR PROJECT MAIN CONFIG — Two Latest 2026 Models Only
# ============================================================================
results:
  directory: "Results/Minor_Project_Main"
  keep_best_worst_checkpoints_only: True
  checkpoint_pruning_metric: "OA"

dataset:
  names: ["Indian_Pines", "Pavia University"]
  run_all_datasets: False
  patch_size: 11
  stride: 1
  verbose: True

model:
  name: [
    "MMFormer",         # 2026 Transformer (IEEE TGRS)
    "R2Mamba"           # 2026 Mamba / SSM (IEEE JSTARS)
  ]
  run_all_models: False
  exclude: []
  print_summary: True
  summary_only: False
  summary_depth: 1

data_split:
  method: "samples"
  disjoint: False
  split_samples: [30, 10]         # 30 train / 10 val per class for Pavia University
  split_samples_overrides:
    Indian_Pines: [10, 5]         # Indian Pines has small classes
  split_ratios: null
  random_state: 42
  seeds: [1, 2, 3]               # 3 seeded runs -> genuine Mean ± Std
  print_stats: True

preprocessing:
  dim_reduction_method: pca
  num_pca_bands: 30
  maxpool_kernel: 2
  use_channel_dim: True
  band_indices: null

training:
  num_epochs: 80                 # 80 epochs with early stopping at 12
  num_runs: 3                    # 3 runs per model
  batch_size: 64                 # Optimal for GPU
  learning_rate: 0.001
  optimizer: "adam"
  optimizer_params: {}
  patience: 12                   # Early stops once converged
  checkpoint_interval: 10
  num_workers: 2

device:
  use_cuda: True
  cuda_device: 0

visualization:
  generate_maps: True
  mode: "labeled_only"
  use_spy_colors: True
  block_background: True
  show_colorbar: False
  dpi: 300

map_arrangement:
  enabled: True
  preset: "paper"
  base_output_dir: "Results/Minor_Project_Main"
  all_datasets: True
  all_models: True
  map_type: ["best"]
  metric: ["OA"]
  visualization:
    mode: "labeled_only"
    use_spy_colors: True
    block_background: True
    include_legend: False
  layout:
    figure_width: 7.16
    gap: 3

score_arrangement:
  all_datasets: True
  all_models: True
  models: []
  use_mean: True
"""

with open("config/project_main.yaml", "w") as f:
    f.write(config)

print("✅ Configuration saved to config/project_main.yaml")

### 🔍 Cell 6: Confirm All Target Models are Loaded

In [ ]:
!python main.py --list-models

### 🚀 Cell 7: Execute Full Experiment Benchmark Training

> **Note:** This trains all 7 architectures over 3 runs on both Indian Pines and Pavia University.  
> Thanks to early stopping (`patience: 12`), each run typically completes in 15–25 epochs.

In [ ]:
!python main.py config/project_main.yaml

### 📊 Cell 8: Generate Academic LaTeX Tables (Mean ± Std)

In [ ]:
!python main.py config/project_main.yaml --arrange-scores

### 🗺️ Cell 9: Arrange Paper-Ready Multi-Model Classification Map Figure

In [ ]:
!python main.py config/project_main.yaml --arrange-only

### 📋 Cell 10: View All Metrics Summary Table

In [ ]:
import glob
import csv
from tabulate import tabulate

csvs = sorted(glob.glob("Results/Minor_Project_Main/**/results_summary.csv", recursive=True))
print(f"Found {len(csvs)} results summaries across models & datasets:\n")

for csv_path in csvs:
    parts = csv_path.split("/")
    dataset_name = parts[-3]
    model_name = parts[-2]
    with open(csv_path, mode="r") as f:
        rows = list(csv.reader(f))
    
    if len(rows) > 1:
        print(f"\n📊 Dataset: {dataset_name} | Model: {model_name}")
        print(tabulate(rows[1:], headers=rows[0], tablefmt="fancy_grid"))

### 🖼️ Cell 11: Display Combined Classification Maps

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image

# Look for arranged comparison maps first
arranged_maps = glob.glob("Results/Minor_Project_Main/*arranged*.png") + glob.glob("Results/Minor_Project_Main/**/*arranged*.png", recursive=True)

if arranged_maps:
    for m in arranged_maps:
        print(f"Displaying: {m}")
        img = Image.open(m)
        plt.figure(figsize=(14, 8))
        plt.imshow(img)
        plt.axis("off")
        plt.show()
else:
    print("Displaying individual best run maps:")
    maps = glob.glob("Results/Minor_Project_Main/**/classification_map.png", recursive=True)
    for m in maps[:6]:
        img = Image.open(m)
        plt.figure(figsize=(5, 5))
        plt.imshow(img)
        plt.title(" / ".join(m.split("/")[-3:-1]))
        plt.axis("off")
        plt.show()

### 📦 Cell 12: Package Final Results for Download

> Creates a single ZIP archive containing all checkpoints, logs, tables, and publication maps.

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/HSI_Minor_Project_Final_Results", "zip", "Results/Minor_Project_Main")
print("✅ Saved as /kaggle/working/HSI_Minor_Project_Final_Results.zip")
print("Download this file from the Kaggle 'Output' files panel on the right sidebar!")